<a href="https://colab.research.google.com/github/YusufAbdil03/DSA210_TermProject-2025-2026_Spring-/blob/main/codes/Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytrends
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pytrends.request import TrendReq

In [ ]:
leagues = {
    'Premier League': 'GB1',
    'LaLiga': 'ES1',
    'Bundesliga': 'L1',
    'Serie A': 'IT1',
    'Ligue 1': 'FR1',
    'Süper Lig': 'TR1'
}
years = range(2015, 2025)
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36'}
pytrends = TrendReq(hl='tr-TR', tz=360)
all_data_list = []

for league_name, league_id in leagues.items():
    print(f"--- {league_name} Başladı ---")
    try:
        pytrends.build_payload([league_name], timeframe='2015-01-01 2025-01-01', geo='TR')
        g_data = pytrends.interest_over_time()
        g_annual = g_data.resample('YE').mean() if not g_data.empty else pd.DataFrame()
        if not g_annual.empty: g_annual.index = g_annual.index.year
    except:
        g_annual = pd.DataFrame()

    for year in years:
        mv_url = f"https://www.transfermarkt.com.tr/direktlink/startseite/wettbewerb/{league_id}/plus/?saison_id={year}"
        att_url = f"https://www.transfermarkt.com.tr/direktlink/besucherzahlen/wettbewerb/{league_id}/plus/?saison_id={year}"

        for attempt in range(3):
            try:
                mv_res = requests.get(mv_url, headers=headers, timeout=20)
                att_res = requests.get(att_url, headers=headers, timeout=20)

                if mv_res.status_code == 200 and att_res.status_code == 200:
                    mv_soup = BeautifulSoup(mv_res.content, 'html.parser')
                    att_soup = BeautifulSoup(att_res.content, 'html.parser')

                    footer = att_soup.find("tfoot")
                    avg_att = int(footer.find_all("td")[-2].text.strip().replace('.', '')) if footer else 0
                    google_score = g_annual.loc[year].values[0] if (not g_annual.empty and year in g_annual.index) else 0

                    mv_table = mv_soup.find("table", {"class": "items"})
                    if mv_table:
                        rows = mv_table.find_all("tr", {"class": ["odd", "even"]})
                        for r in rows:
                            c = r.find_all("td")
                            if len(c) > 6:
                                team = c[2].text.strip()
                                txt = c[6].text.strip().lower()
                                val = float(txt.split(' ')[0].replace(',', '.')) * 1000 if 'milyar' in txt else (float(txt.split(' ')[0].replace(',', '.')) if 'mil' in txt else 0)

                                all_data_list.append({
                                    'Year': year, 'League': league_name, 'Team': team,
                                    'MarketValue_M_Euro': val, 'League_Avg_Attendance': avg_att,
                                    'League_Google_Interest': google_score
                                })
                    print(f"✅ {league_name} {year} Tamamlandı")
                    break
                else:
                    print(f"⚠️ {league_name} {year} Durum Kodu: {mv_res.status_code}. Tekrar deneniyor...")
            except Exception as e:
                print(f"🔄 {league_name} {year} deneme {attempt+1} başarısız: {e}")
                time.sleep(5)

        time.sleep(3)

final_df = pd.DataFrame(all_data_list)
for l in leagues.keys():
    mask_2020 = (final_df['League'] == l) & (final_df['Year'] == 2020)
    val_2019 = final_df[(final_df['League'] == l) & (final_df['Year'] == 2019)]['League_Avg_Attendance'].unique()
    val_2021 = final_df[(final_df['League'] == l) & (final_df['Year'] == 2021)]['League_Avg_Attendance'].unique()
    if len(val_2019) > 0 and len(val_2021) > 0:
        final_df.loc[mask_2020, 'League_Avg_Attendance'] = (val_2019[0] + val_2021[0]) / 2

final_df.to_csv('final_comprehensive_football_data.csv', index=False)
from google.colab import files
files.download('final_comprehensive_football_data.csv')
print("🔥 EKSİKSİZ VERİ SETİ HAZIR!")

--- Premier League Başladı ---
✅ Premier League 2015 Tamamlandı
✅ Premier League 2016 Tamamlandı
✅ Premier League 2017 Tamamlandı
✅ Premier League 2018 Tamamlandı
✅ Premier League 2019 Tamamlandı
✅ Premier League 2020 Tamamlandı
✅ Premier League 2021 Tamamlandı
✅ Premier League 2022 Tamamlandı
✅ Premier League 2023 Tamamlandı
✅ Premier League 2024 Tamamlandı
--- LaLiga Başladı ---
✅ LaLiga 2015 Tamamlandı
✅ LaLiga 2016 Tamamlandı
✅ LaLiga 2017 Tamamlandı
✅ LaLiga 2018 Tamamlandı
✅ LaLiga 2019 Tamamlandı
✅ LaLiga 2020 Tamamlandı
✅ LaLiga 2021 Tamamlandı
✅ LaLiga 2022 Tamamlandı
✅ LaLiga 2023 Tamamlandı
✅ LaLiga 2024 Tamamlandı
--- Bundesliga Başladı ---
✅ Bundesliga 2015 Tamamlandı
✅ Bundesliga 2016 Tamamlandı
✅ Bundesliga 2017 Tamamlandı
✅ Bundesliga 2018 Tamamlandı
✅ Bundesliga 2019 Tamamlandı
🔄 Bundesliga 2020 deneme 1 başarısız: HTTPSConnectionPool(host='www.transfermarkt.com.tr', port=443): Read timed out. (read timeout=20)
✅ Bundesliga 2020 Tamamlandı
✅ Bundesliga 2021 Tamamlandı


/tmp/ipykernel_6934/2843069672.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '8296552.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[mask_2020, 'League_Avg_Attendance'] = (val_2019[0] + val_2021[0]) / 2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🔥 EKSİKSİZ VERİ SETİ HAZIR!


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

df = pd.read_csv('final_comprehensive_football_data.csv')
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36'}
leagues = {'Premier League': 'GB1', 'LaLiga': 'ES1', 'Bundesliga': 'L1', 'Serie A': 'IT1', 'Ligue 1': 'FR1', 'Süper Lig': 'TR1'}

def get_names(l_id):
    url = f"https://www.transfermarkt.com.tr/direktlink/startseite/wettbewerb/{l_id}/plus/?saison_id=2023"
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.content, 'html.parser')
    rows = soup.find("table", {"class": "items"}).find_all("tr", {"class": ["odd", "even"]})
    return [r.find("td", {"class": "hauptlink"}).find("a").text.strip() for r in rows]

df['Team_Name'] = ""
for l_name, l_id in leagues.items():
    print(f"{l_name} isimleri eşleşiyor...")
    correct_names = get_names(l_id)
    league_mask = df['League'] == l_name
    unique_ids = df[league_mask]['Team'].unique()
    id_to_name = {uid: name for uid, name in zip(unique_ids, correct_names)}
    df.loc[league_mask, 'Team_Name'] = df.loc[league_mask, 'Team'].map(id_to_name)

df.to_csv('final_comprehensive_football_data_fixed.csv', index=False)
from google.colab import files
files.download('final_comprehensive_football_data_fixed.csv')

Premier League isimleri eşleşiyor...
LaLiga isimleri eşleşiyor...
Bundesliga isimleri eşleşiyor...
Serie A isimleri eşleşiyor...
Ligue 1 isimleri eşleşiyor...
Süper Lig isimleri eşleşiyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

# 1. Dosyayı oku
df = pd.read_csv('final_comprehensive_football_data_fixed.csv')

# 2. Lig + Team ID bazlı eşleştirme (En güvenli yol)
# Boş olmayan isimlerden bir "nüfus cüzdanı" defteri oluşturuyoruz
id_name_map = df[df['Team_Name'].notna()].drop_duplicates(['League', 'Team']).set_index(['League', 'Team'])['Team_Name'].to_dict()

# 3. Boşlukları (NaN) bu rehbere göre doldur
def fill_names(row):
    if pd.isna(row['Team_Name']) or row['Team_Name'] == "":
        return id_name_map.get((row['League'], row['Team']), f"Team_{row['Team']}")
    return row['Team_Name']

df['Team_Name'] = df.apply(fill_names, axis=1)

# 4. Kaydet ve indir
df.to_csv('final_football_data_v2.csv', index=False)
from google.colab import files
files.download('final_football_data_v2.csv')

print("Lig ve ID kontrolü yapıldı, isimler mermi gibi çakıldı!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Lig ve ID kontrolü yapıldı, isimler mermi gibi çakıldı!
